# EDA Market Index Dataset (2010-2025)
- Load all market index datasets from the `dataset` folder.
- Keep observations from 2010-01-01 to 2025-12-31 only.
- Create **Log Return** only (no simple return):
  $$\log\_return_t = \ln\left(\frac{Close_t}{Close_{t-1}}\right)$$
- Produce descriptive statistics and risk-oriented indicators for each dataset.
- Build an interactive dashboard to explore and compare two datasets by distribution of Close or Log Return.

In [1]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import Markdown, display

warnings.filterwarnings("ignore")
pd.options.display.float_format = "{:,.6f}".format

START_DATE = pd.Timestamp("2010-01-01")
END_DATE = pd.Timestamp("2025-12-31")
VOL_WINDOW = 60

In [2]:
def find_project_root() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "dataset").exists():
            return candidate
    raise FileNotFoundError("Cannot find the 'dataset' folder from current working directory.")


PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "dataset"


def load_market_dataset(csv_path: Path, start_date: pd.Timestamp, end_date: pd.Timestamp, vol_window: int = VOL_WINDOW) -> pd.DataFrame:
    df = pd.read_csv(csv_path)
    df.columns = [c.strip().lower() for c in df.columns]

    required = {"time", "close"}
    missing = required.difference(df.columns)
    if missing:
        raise ValueError(f"{csv_path.name} is missing columns: {sorted(missing)}")

    df["time"] = pd.to_datetime(df["time"], errors="coerce")
    df["close"] = pd.to_numeric(df["close"], errors="coerce")

    df = (
        df[["time", "close"]]
        .dropna(subset=["time", "close"])
        .sort_values("time")
        .drop_duplicates(subset=["time"], keep="last")
    )

    df = df.loc[(df["time"] >= start_date) & (df["time"] <= end_date)].copy()

    # Log return in decimal form, then scale to percent-point like benchmark pipeline.
    df["log_return"] = np.log(df["close"] / df["close"].shift(1))
    df["log_return_pp"] = df["log_return"] * 100.0

    # True volatility proxy: use only past returns up to t-1 (shift(1)).
    df["volatility"] = (
        df["log_return_pp"]
        .shift(1)
        .rolling(window=int(vol_window))
        .std(ddof=0)
    )

    df = df.replace([np.inf, -np.inf], np.nan)

    return df.reset_index(drop=True)


market_data = {}
for csv_file in sorted(DATA_DIR.glob("*.csv")):
    market_data[csv_file.stem] = load_market_dataset(csv_file, START_DATE, END_DATE)

if not market_data:
    raise RuntimeError(f"No CSV files were loaded from {DATA_DIR}")

coverage_rows = []
for name, df in market_data.items():
    coverage_rows.append(
        {
            "dataset": name,
            "obs_close": int(df["close"].notna().sum()),
            "obs_log_return": int(df["log_return"].notna().sum()),
            "obs_volatility": int(df["volatility"].notna().sum()),
            "start": df["time"].min(),
            "end": df["time"].max(),
        }
    )

coverage_df = pd.DataFrame(coverage_rows).sort_values("dataset").reset_index(drop=True)
display(Markdown(f"**Data folder:** `{DATA_DIR}`"))
coverage_df

**Data folder:** `D:\UIT\1003_EPA_PROJECT\1003_EPA-Project_UIT\dataset`

,dataset,obs_close,obs_log_return,obs_volatility,start,end
0,DAX_40,4059,4058,3998,2010-01-04,2025-12-30
1,EuroNext_100,4099,4098,4038,2010-01-04,2025-12-31
2,IBEX_35,4100,4099,4039,2010-01-04,2025-12-31
3,KOSPI_index,3934,3933,3873,2010-01-04,2025-12-30
4,Nikkei_225,3913,3912,3852,2010-01-04,2025-12-30
5,SMI,4023,4022,3962,2010-01-04,2025-12-30
6,VN30_INDEX,3992,3991,3931,2010-01-04,2025-12-31
7,VN_INDEX,3992,3991,3931,2010-01-04,2025-12-31
8,snp500,4024,4023,3963,2010-01-04,2025-12-31


## Indicators and methods used in this EDA
For each market index, the notebook reports:
- Descriptive statistics for `close` and `log_return` (count, mean, std, quantiles, skewness, kurtosis).
- Risk and variability indicators from `log_return`:
  - Daily volatility (std of log return).
  - Annualized volatility: $$\sigma_{annual} = \sigma_{daily}\sqrt{252}$$
  - Historical VaR at 95% and 99% levels.
- Drawdown indicator from `close`:
  - Running maximum and maximum drawdown over the selected period.

In [3]:
def compute_dataset_metrics(df: pd.DataFrame) -> dict:
    close = df["close"].dropna()
    log_return = df["log_return"].dropna()

    if close.empty:
        return {
            "n_close": 0,
            "n_log_return": 0,
            "start": pd.NaT,
            "end": pd.NaT,
        }

    rolling_max = close.cummax()
    drawdown = (close / rolling_max) - 1.0

    metrics = {
        "n_close": int(close.count()),
        "n_log_return": int(log_return.count()),
        "start": df["time"].min(),
        "end": df["time"].max(),

        "close_mean": close.mean(),
        "close_std": close.std(),
        "close_min": close.min(),
        "close_q05": close.quantile(0.05),
        "close_median": close.median(),
        "close_q95": close.quantile(0.95),
        "close_max": close.max(),

        "logret_mean": log_return.mean(),
        "logret_std": log_return.std(),
        "logret_skew": log_return.skew(),
        "logret_kurt": log_return.kurt(),
        "logret_min": log_return.min(),
        "logret_q01": log_return.quantile(0.01),
        "logret_q05": log_return.quantile(0.05),
        "logret_median": log_return.median(),
        "logret_q95": log_return.quantile(0.95),
        "logret_q99": log_return.quantile(0.99),
        "logret_max": log_return.max(),

        "daily_volatility": log_return.std(),
        "annualized_volatility": log_return.std() * np.sqrt(252),
        "VaR_95": log_return.quantile(0.05),
        "VaR_99": log_return.quantile(0.01),
        "max_drawdown": drawdown.min(),
    }
    return metrics


metrics_df = pd.DataFrame({name: compute_dataset_metrics(df) for name, df in market_data.items()}).T
metrics_df = metrics_df.sort_index()
metrics_df

,n_close,n_log_return,start,end,close_mean,close_std,close_min,close_q05,close_median,close_q95,...,logret_q05,logret_median,logret_q95,logret_q99,logret_max,daily_volatility,annualized_volatility,VaR_95,VaR_99,max_drawdown
DAX_40,4059,4058,2010-01-04 00:00:00,2025-12-30 00:00:00,"12,132.654959","4,452.310491","5,072.330078","6,057.830908","11,955.250000","22,327.006445",...,-0.019425,0.000804,0.018302,0.031137,0.104143,0.012242,0.194336,-0.019425,-0.034838,-0.387794
EuroNext_100,4099,4098,2010-01-04 00:00:00,2025-12-31 00:00:00,"1,020.554877",290.199285,529.500000,620.367000,990.990000,"1,554.111000",...,-0.017798,0.000630,0.016707,0.029275,0.081403,0.011210,0.177955,-0.017798,-0.033435,-0.379130
IBEX_35,4100,4099,2010-01-04 00:00:00,2025-12-31 00:00:00,"9,654.323341","1,722.307949","5,956.300000","7,174.305000","9,394.750000","13,013.100000",...,-0.020468,0.000536,0.019400,0.033631,0.134836,0.013381,0.212416,-0.020468,-0.036637,-0.512677
KOSPI_index,3934,3933,2010-01-04 00:00:00,2025-12-30 00:00:00,"2,276.757176",437.085028,"1,457.640015","1,776.842554","2,119.484985","3,161.876953",...,-0.016954,0.000484,0.016090,0.026587,0.082513,0.010823,0.171808,-0.016954,-0.030956,-0.438979
Nikkei_225,3913,3912,2010-01-04 00:00:00,2025-12-30 00:00:00,"21,714.393106","9,622.172828","8,160.009766","8,922.031836","20,556.539062","39,369.439063",...,-0.020933,0.000728,0.020361,0.030794,0.097366,0.013415,0.212959,-0.020933,-0.036718,-0.317989
SMI,4023,4022,2010-01-04 00:00:00,2025-12-30 00:00:00,"9,257.509861","2,005.304938","4,791.960000","6,101.649000","9,024.320000","12,326.642000",...,-0.014496,0.000566,0.013846,0.023476,0.067805,0.009515,0.151053,-0.014496,-0.026751,-0.312247
VN30_INDEX,3992,3991,2010-01-04 00:00:00,2025-12-31 00:00:00,874.181951,365.363838,374.020000,456.837000,790.590000,"1,516.449000",...,-0.019452,0.001039,0.018438,0.030956,0.066700,0.012235,0.194221,-0.019452,-0.040547,-0.481387
VN_INDEX,3992,3991,2010-01-04 00:00:00,2025-12-31 00:00:00,853.385521,350.522991,336.730000,410.658000,829.045000,"1,469.428500",...,-0.019042,0.001041,0.017341,0.029282,0.065469,0.011794,0.187228,-0.019042,-0.038546,-0.452633
snp500,4024,4023,2010-01-04 00:00:00,2025-12-31 00:00:00,"2,945.011381","1,491.073720","1,022.580017","1,173.834045","2,584.730103","5,955.137500",...,-0.016605,0.000702,0.015408,0.026214,0.090895,0.010942,0.173704,-0.016605,-0.032208,-0.339250


## Interactive dashboard
Use the controls to:
- switch between single-dataset and two-dataset comparison,
- choose feature distribution (`Close`, `Log Return`, or `Volatility`),
- adjust date range and histogram bins.

In [4]:
dataset_names = sorted(market_data.keys())
feature_map = {"Close": "close", "Log Return": "log_return", "Volatility": "volatility"}

mode_widget = widgets.ToggleButtons(
    options=["Single", "Compare"],
    value="Single",
    description="Mode:",
    style={"button_width": "110px"},
)

dataset_a_widget = widgets.Dropdown(
    options=dataset_names,
    value=dataset_names[0],
    description="Dataset A:",
)

dataset_b_widget = widgets.Dropdown(
    options=[("None", None)] + [(name, name) for name in dataset_names],
    value=None,
    description="Dataset B:",
)

feature_widget = widgets.Dropdown(
    options=list(feature_map.keys()),
    value="Close",
    description="Feature:",
)

bins_widget = widgets.IntSlider(
    value=60,
    min=20,
    max=180,
    step=5,
    description="Bins:",
)

start_picker = widgets.DatePicker(
    description="Start:",
    value=START_DATE.date(),
)

end_picker = widgets.DatePicker(
    description="End:",
    value=END_DATE.date(),
)

output_plot = widgets.Output()
output_table = widgets.Output()


def _safe_date(value, fallback):
    if value is None:
        return fallback
    return pd.Timestamp(value)


def _selected_markets():
    names = [dataset_a_widget.value]
    if mode_widget.value == "Compare" and dataset_b_widget.value is not None:
        names.append(dataset_b_widget.value)

    # Keep order but remove duplicates
    result = []
    for name in names:
        if name not in result:
            result.append(name)
    return result


def _filter_by_dates(df: pd.DataFrame) -> pd.DataFrame:
    start_ts = _safe_date(start_picker.value, START_DATE)
    end_ts = _safe_date(end_picker.value, END_DATE)

    if start_ts > end_ts:
        start_ts, end_ts = end_ts, start_ts

    return df.loc[(df["time"] >= start_ts) & (df["time"] <= end_ts)].copy()


def _feature_summary(series: pd.Series, dataset_name: str) -> dict:
    s = series.dropna()
    if s.empty:
        return {
            "dataset": dataset_name,
            "n": 0,
            "mean": np.nan,
            "std": np.nan,
            "min": np.nan,
            "q05": np.nan,
            "median": np.nan,
            "q95": np.nan,
            "max": np.nan,
            "skew": np.nan,
            "kurt": np.nan,
        }

    return {
        "dataset": dataset_name,
        "n": int(s.count()),
        "mean": s.mean(),
        "std": s.std(),
        "min": s.min(),
        "q05": s.quantile(0.05),
        "median": s.median(),
        "q95": s.quantile(0.95),
        "max": s.max(),
        "skew": s.skew(),
        "kurt": s.kurt(),
    }


def update_dashboard(*_):
    feature_col = feature_map[feature_widget.value]
    selected_markets = _selected_markets()

    fig = go.Figure()
    summary_rows = []

    for name in selected_markets:
        df = _filter_by_dates(market_data[name])
        series = df[feature_col].dropna()

        summary_rows.append(_feature_summary(series, name))

        if not series.empty:
            fig.add_trace(
                go.Histogram(
                    x=series,
                    name=name,
                    nbinsx=bins_widget.value,
                    opacity=0.65,
                    histnorm="probability density",
                )
            )

    fig.update_layout(
        template="plotly_white",
        barmode="overlay",
        title=f"Distribution of {feature_widget.value}",
        xaxis_title=feature_widget.value,
        yaxis_title="Density",
        legend_title="Dataset",
        height=500,
    )

    with output_plot:
        output_plot.clear_output(wait=True)
        display(fig)

    summary_df = pd.DataFrame(summary_rows).set_index("dataset") if summary_rows else pd.DataFrame()

    with output_table:
        output_table.clear_output(wait=True)
        if summary_df.empty:
            display(Markdown("No data available for the selected options."))
        else:
            display(summary_df)


for w in [
    mode_widget,
    dataset_a_widget,
    dataset_b_widget,
    feature_widget,
    bins_widget,
    start_picker,
    end_picker,
]:
    w.observe(update_dashboard, names="value")

controls_row_1 = widgets.HBox([mode_widget, feature_widget, bins_widget])
controls_row_2 = widgets.HBox([dataset_a_widget, dataset_b_widget])
controls_row_3 = widgets.HBox([start_picker, end_picker])

dashboard_ui = widgets.VBox([
    controls_row_1,
    controls_row_2,
    controls_row_3,
    output_plot,
    output_table,
])

update_dashboard()
display(dashboard_ui)